# SHAP Explainability

## CircuitBench Tutorial 7

This notebook demonstrates publication-quality explainable artificial intelligence (XAI) analysis using SHAP (SHapley Additive exPlanations) to interpret surrogate machine learning models for electronic circuit benchmarking.

### Learning Objectives

- Load a trained surrogate model
- Compute SHAP values
- Quantify feature importance
- Explain individual predictions
- Generate global explanations
- Produce publication-quality visualizations
- Export explainability reports


## Scientific Background

SHAP is a game-theoretic explainability framework that assigns each feature a contribution value for every prediction. It provides both global model interpretation and local explanation of individual predictions while satisfying desirable theoretical properties such as consistency and local accuracy.

In [ ]:
from pathlib import Path
import warnings
import random

import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
import shap

from circuitbench.benchmark import BenchmarkRunner

## Load Benchmark Dataset

In [ ]:
runner = BenchmarkRunner()

benchmark = runner.load("CMOS_Inverter")

## Load Trained Model

In [ ]:
model = runner.load_model("RandomForest")

## Prepare Evaluation Data

In [ ]:
X = benchmark.X_test
y = benchmark.y_test

feature_names = benchmark.feature_names

## Initialize SHAP Explainer

In [ ]:
explainer = shap.Explainer(model)

shap_values = explainer(X)

## SHAP Value Dimensions

In [ ]:
print("Samples :", shap_values.values.shape[0])
print("Features:", shap_values.values.shape[1])

## Global Feature Importance

In [ ]:
shap.plots.bar(shap_values, max_display=20, show=False)
plt.tight_layout()

## SHAP Summary Plot

In [ ]:
shap.summary_plot(shap_values, X, feature_names=feature_names, show=False)
plt.tight_layout()

## Mean Absolute SHAP Values

In [ ]:
importance = pd.DataFrame(
    {"Feature": feature_names, "MeanAbsSHAP": np.abs(shap_values.values).mean(axis=0)}
)

importance = importance.sort_values(by="MeanAbsSHAP", ascending=False).reset_index(
    drop=True
)

importance

## Top 10 Most Important Features

In [ ]:
top10 = importance.head(10)
top10

## Publication Quality Importance Plot

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(top10["Feature"][::-1], top10["MeanAbsSHAP"][::-1])
plt.xlabel("Mean |SHAP Value|")
plt.ylabel("Feature")
plt.title("Top 10 Most Important Features")
plt.tight_layout()

## Save Feature Importance

In [ ]:
output_dir = Path("shap_results")
output_dir.mkdir(exist_ok=True)

importance.to_csv(output_dir / "feature_importance.csv", index=False)

## Local Explanation

In [ ]:
sample_index = 0

shap.plots.waterfall(shap_values[sample_index], max_display=15, show=False)
plt.tight_layout()

## Force Plot

In [ ]:
shap.initjs()

shap.force_plot(
    explainer.expected_value,
    shap_values.values[sample_index],
    X[sample_index],
    feature_names=feature_names,
)

## Dependence Plot

In [ ]:
top_feature = importance.iloc[0]["Feature"]

shap.dependence_plot(
    top_feature, shap_values.values, X, feature_names=feature_names, show=False
)
plt.tight_layout()

## SHAP Value Statistics

In [ ]:
statistics = pd.DataFrame(
    {
        "Feature": feature_names,
        "Mean": np.mean(shap_values.values, axis=0),
        "Std": np.std(shap_values.values, axis=0),
        "Minimum": np.min(shap_values.values, axis=0),
        "Maximum": np.max(shap_values.values, axis=0),
    }
)

statistics.head()

## Export SHAP Statistics

In [ ]:
statistics.to_csv(output_dir / "shap_statistics.csv", index=False)

## Export Explainability Report

In [ ]:
report = {
    "method": "SHAP",
    "samples": int(shap_values.values.shape[0]),
    "features": int(shap_values.values.shape[1]),
    "top_feature": str(importance.iloc[0]["Feature"]),
    "mean_importance": float(importance.iloc[0]["MeanAbsSHAP"]),
    "random_seed": SEED,
}
with open(output_dir / "shap_report.json", "w") as f:
    json.dump(report, f, indent=4)

## Reproducibility Checklist

- Fixed random seed used throughout the analysis.
- Same evaluation dataset used for all explanations.
- SHAP version documented.
- Feature importance exported.
- Local explanations archived.
- Explainability statistics saved.
- JSON report generated for reproducibility.


## Best Practices

1. Report both global and local explanations.
2. Compare SHAP importance with permutation importance when appropriate.
3. Interpret explanations alongside predictive performance.
4. Use the same preprocessing pipeline during training and explanation.
5. Export all explainability artifacts for reproducible research.


## Citation

If this workflow contributes to published research, please cite the CircuitBench framework together with the original SHAP publication.

# Summary

In this notebook we:

- Computed SHAP values for a trained surrogate model.
- Ranked features according to their global importance.
- Generated summary, bar, waterfall, force, and dependence plots.
- Explained individual model predictions.
- Exported explainability statistics and publication-ready reports.

These analyses provide transparent and reproducible interpretation of machine learning models within CircuitBench.